 # **Import Libraries**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.cluster import KMeans
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, KFold
from scipy.stats import shapiro, f_oneway, stats
from statsmodels.stats.multicomp import MultiComparison
from yellowbrick.regressor import ResidualsPlot


 # **Regression algorithms for farm areas**





In [ ]:
# Load data
all_areas_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/files/output/final/data_integration_all_areas_final.csv")


In [ ]:
all_areas_df.select_dtypes(include=['int64','float64'])


In [ ]:

# 🚀 SETUP GLOBAL PARA xAI E PREVENÇÃO DE VAZAMENTO DE DADOS
features_xai = [
    'area_cod',     # Uso do Solo
    'area_size',    # Tamanho da Área
    'city_cod',     # Cidade
    'state_cod',    # Estado
    'biome_cod',    # Bioma
    'climate_cod'   # Clima
]
target = 'balance_CO2_ha'

# Correlation APENAS com as features permitidas e o target (Sem Data Leakage)
colunas_correlacao = features_xai + [target]
all_areas_df_corr = all_areas_df[colunas_correlacao]

figura = plt.figure(figsize=(18,8))
sns.heatmap(all_areas_df_corr.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.show()


In [ ]:
all_areas_df.count()


 #**Linear Regression - Simple**

In [ ]:
# X is the single predictor attribute: climate_cod
X_areas = all_areas_df['climate_cod'].values
X_areas


In [ ]:
# Y is the attribute: balance_CO2_ha
Y_areas = all_areas_df[target].values
Y_areas


In [ ]:
X_areas.shape, Y_areas.shape


In [ ]:
# Correlation coefficient
np.corrcoef(X_areas, Y_areas)


In [ ]:
# Transforming array into matrix (required for sklearn)
X_areas = X_areas.reshape(-1,1)
X_areas


In [ ]:
X_areas.shape


 **Standardization, if necessary**

In [ ]:
# This prevents scaling bias and ensures stable training for neural networks and polynomial models.
scaler_areas_x = StandardScaler()
X_areas = scaler_areas_x.fit_transform((X_areas).reshape(-1,1))
scaler_areas_y = StandardScaler()
Y_areas = scaler_areas_y.fit_transform((Y_areas).reshape(-1,1))
X_areas, Y_areas


 **Using the entire database**

In [ ]:
# Create the regression model (for all data)
simple_linear_regressor = LinearRegression()
simple_linear_regressor.fit(X_areas, Y_areas)


In [ ]:
# b0 : beginning of the regression line and b1: slope of the line
simple_linear_regressor.intercept_, simple_linear_regressor.coef_


In [ ]:
# Prevision
prevision_simple_linear_regressor = simple_linear_regressor.predict(X_areas)
prevision_simple_linear_regressor


In [ ]:
# Convert matrix to array (for the graph)
X_areas.ravel()


In [ ]:
# Prediction with a standardized climate code (value 15) = estimated value of carbon/ha
simple_linear_regressor.intercept_ + scaler_areas_x.transform([[15]])[0][0]


In [ ]:
# Scale the raw value before passing it to the predict function
raw_value = [[15]]
scaled_value = scaler_areas_x.transform(raw_value)

simple_linear_regressor.predict(scaled_value)


In [ ]:
# score: algorithm quality metric (closest to 1 best value)
simple_linear_regressor.score(X_areas, Y_areas)


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas, prevision_simple_linear_regressor)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas, prevision_simple_linear_regressor)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas, prevision_simple_linear_regressor))


In [ ]:
# distance from the original values to the linear regression line (Train R2 = algorithm quality)
viewer = ResidualsPlot(simple_linear_regressor)
viewer.fit(X_areas, Y_areas)
viewer.poof()


In [ ]:
# Correlation viewer
figure = plt.figure(figsize=(20,5))
sns.heatmap(all_areas_df_corr.corr(), annot=True)


 **Simple Linear Regression - training and testing bases**

In [ ]:
# Division of bases (75% trein, 25% test)
X_areas_trein, X_areas_test, Y_areas_trein, Y_areas_test = train_test_split(X_areas, Y_areas, test_size = 0.25, random_state = 0)


In [ ]:
X_areas_trein.shape, Y_areas_trein.shape, X_areas_test.shape, Y_areas_test.shape




 **Standardization, if necessary**

In [ ]:
# We fit and transform the training set, but only transform the test set 
# using the exact parameters (mean and variance) learned from the training data.
scaler_areas_x = StandardScaler()
X_areas_trein = scaler_areas_x.fit_transform((X_areas_trein).reshape(-1,1))
X_areas_test = scaler_areas_x.transform((X_areas_test).reshape(-1,1))

scaler_areas_y = StandardScaler()
Y_areas_trein = scaler_areas_y.fit_transform((Y_areas_trein).reshape(-1,1))
Y_areas_test = scaler_areas_y.transform((Y_areas_test).reshape(-1,1))

X_areas_trein, Y_areas_trein, X_areas_test, Y_areas_test



 **Regression Model for training**

In [ ]:
# Convert array to matrix
X_areas_trein = X_areas_trein.reshape(-1,1)
X_areas_trein


In [ ]:
# Create the regression model (trein)
simple_linear_regressor_trein = LinearRegression()
simple_linear_regressor_trein.fit(X_areas_trein, Y_areas_trein)


In [ ]:
# score: algorithm quality metric (trein)
simple_linear_regressor_trein.score(X_areas_trein, Y_areas_trein)


In [ ]:
# Prevision (trein)
prevision_simple_linear_regressor_trein = simple_linear_regressor_trein.predict(X_areas_trein)
prevision_simple_linear_regressor_trein


In [ ]:
# Real data
Y_areas_trein


In [ ]:
# Mean absolute error
abs(Y_areas_trein-prevision_simple_linear_regressor_trein).mean()


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_trein, prevision_simple_linear_regressor_trein)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_trein, prevision_simple_linear_regressor_trein)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_trein, prevision_simple_linear_regressor_trein))


In [ ]:
# Prediction with a standardized climate code (value 15) = estimated value of carbon/ha
simple_linear_regressor_trein.intercept_ + scaler_areas_x.transform([[15]])[0][0]


 **Regression Model for testing**

In [ ]:
# Convert array to matrix
X_areas_test = X_areas_test.reshape(-1,1)
X_areas_test


In [ ]:
# CORRIGIDO: Removida a criação e treino de um novo modelo na base de teste
# score: algorithm quality metric (test)
simple_linear_regressor_trein.score(X_areas_test, Y_areas_test)


In [ ]:
# Prevision (test)
prevision_simple_linear_regressor_test = simple_linear_regressor_trein.predict(X_areas_test)
prevision_simple_linear_regressor_test


In [ ]:
# Real data
Y_areas_test


In [ ]:
# Mean absolute error
abs(Y_areas_test-prevision_simple_linear_regressor_test).mean()


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_test, prevision_simple_linear_regressor_test)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_test, prevision_simple_linear_regressor_test)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_test, prevision_simple_linear_regressor_test))


In [ ]:
# Prediction with a value  (area 15) = estimated value of carbon/ha
simple_linear_regressor_trein.intercept_ + scaler_areas_x.transform([[15]])[0][0]


 # **Linear Regression - Multiple**

In [ ]:
# X matrix composed of the 6 multi-criteria features (area code, Size, City, State, Biome, Climate)
X_areas_mult = all_areas_df[features_xai].values
X_areas_mult


In [ ]:
# # Y is the attribute: balance_CO2_ha
Y_areas_mult = all_areas_df[target].values
Y_areas_mult


 **Using the entire database**

 **Standardization, if necessary**

In [ ]:
scaler_areas_x = StandardScaler()
X_areas_mult = scaler_areas_x.fit_transform((X_areas_mult))
scaler_areas_y = StandardScaler()
Y_areas_mult = scaler_areas_y.fit_transform((Y_areas_mult).reshape(-1,1))
X_areas_mult, Y_areas_mult


In [ ]:
# Create the regression model (for all data)
mult_linear_regressor = LinearRegression()
mult_linear_regressor.fit(X_areas_mult, Y_areas_mult)


In [ ]:
# Prevision
prevision_mult_linear_regressor = mult_linear_regressor.predict(X_areas_mult)
prevision_mult_linear_regressor


In [ ]:
# score: algorithm quality metric (closest to 1 best value)
mult_linear_regressor.score(X_areas_mult, Y_areas_mult)


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_mult, prevision_mult_linear_regressor)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_mult, prevision_mult_linear_regressor)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_mult, prevision_mult_linear_regressor))


 **Multiple Linear Regression - training and testing bases**

In [ ]:
# Division of bases (75% trein, 25% test)
X_areas_mult_trein, X_areas_mult_test, Y_areas_mult_trein, Y_areas_mult_test = train_test_split(X_areas_mult, Y_areas_mult, test_size = 0.25, random_state = 0)


In [ ]:
X_areas_mult_trein.shape, Y_areas_mult_trein.shape, X_areas_mult_test.shape, Y_areas_mult_test.shape


 **Standardization, if necessary**



In [ ]:
scaler_areas_x_mult = StandardScaler()
X_areas_mult_trein = scaler_areas_x_mult.fit_transform((X_areas_mult_trein))
X_areas_mult_test = scaler_areas_x_mult.transform((X_areas_mult_test)) # CORRIGIDO: Apenas transform no teste

scaler_areas_y_mult = StandardScaler()
Y_areas_mult_trein = scaler_areas_y_mult.fit_transform((Y_areas_mult_trein).reshape(-1, 1)) # CORRIGIDO: fit_transform para evitar NotFittedError
Y_areas_mult_test = scaler_areas_y_mult.transform((Y_areas_mult_test).reshape(-1, 1))

X_areas_mult_trein, Y_areas_mult_trein, X_areas_mult_test, Y_areas_mult_test


 **Regression Model for training**

In [ ]:
# Create the regression model (trein)
mult_linear_regressor_trein = LinearRegression()
mult_linear_regressor_trein.fit(X_areas_mult_trein, Y_areas_mult_trein)


In [ ]:
# score: algorithm quality metric (trein)
mult_linear_regressor_trein.score(X_areas_mult_trein, Y_areas_mult_trein)


In [ ]:
# Prevision (trein)
prevision_mult_linear_regressor_trein = mult_linear_regressor_trein.predict(X_areas_mult_trein)
prevision_mult_linear_regressor_trein


In [ ]:
# Real data
Y_areas_mult_trein


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_mult_trein, prevision_mult_linear_regressor_trein)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_mult_trein, prevision_mult_linear_regressor_trein)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_mult_trein, prevision_mult_linear_regressor_trein))


 **Regression Model for testing**

In [ ]:
# CORRIGIDO: Removida a criação e treino de um novo modelo na base de teste
# score: algorithm quality metric (test)
mult_linear_regressor_trein.score(X_areas_mult_test, Y_areas_mult_test)


In [ ]:
# Prevision (test)
prevision_mult_linear_regressor_test = mult_linear_regressor_trein.predict(X_areas_mult_test)
prevision_mult_linear_regressor_test


In [ ]:
# Real data
Y_areas_mult_test


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_mult_test, prevision_mult_linear_regressor_test)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_mult_test, prevision_mult_linear_regressor_test)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_mult_test, prevision_mult_linear_regressor_test))


 # **Polynomial Regression**

In [ ]:
# X is the single predictor feature: climate_cod
X_areas = all_areas_df['climate_cod'].values
X_areas


In [ ]:
# Y is the attribute: balance_CO2_ha
Y_areas = all_areas_df[target].values
Y_areas


In [ ]:
scaler_polynomial_x = StandardScaler()
X_areas  = scaler_polynomial_x.fit_transform(X_areas.reshape(-1,1))
X_areas


In [ ]:
scaler_polynomial_y = StandardScaler()
Y_areas  = scaler_polynomial_y.fit_transform(Y_areas.reshape(-1,1))
Y_areas


In [ ]:
# Apply degree to the polynomial
polynomial_degree = PolynomialFeatures(degree=2) # CORRIGIDO: Grau abaixado para evitar quebra de memória/overfitting


 **Using the entire database**

In [ ]:
# Transforming array into matrix (required for sklearn)
X_areas = X_areas.reshape(-1,1)
X_areas


In [ ]:
# Apply polynominal degree in X
X_areas_poly = polynomial_degree.fit_transform(X_areas)


In [ ]:
X_areas_poly.shape


In [ ]:
X_areas_poly


In [ ]:
# Create the regression model (for all data)
poly_simple_linear_regressor = LinearRegression()
poly_simple_linear_regressor.fit(X_areas_poly, Y_areas)


In [ ]:
# Score: algorithm quality metric (closest to 1 best value)
poly_simple_linear_regressor.score(X_areas_poly, Y_areas)


In [ ]:
# Prevision
prevision_poly_simple_linear_regressor = poly_simple_linear_regressor.predict(X_areas_poly)
prevision_poly_simple_linear_regressor


In [ ]:
# Real data
Y_areas


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas, prevision_poly_simple_linear_regressor)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas, prevision_poly_simple_linear_regressor)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas, prevision_poly_simple_linear_regressor))


In [ ]:
# Prevision
new_value = [[15]]
new_value = polynomial_degree.transform(new_value)
new_value


In [ ]:
poly_simple_linear_regressor.predict(new_value)


 **Polynomial Linear Regression - training and testing bases**

In [ ]:
# Division of bases (75% trein, 25% test)
X_areas_trein, X_areas_test, Y_areas_trein, Y_areas_test = train_test_split(X_areas, Y_areas, test_size = 0.25, random_state = 0)


 **Regression Model for training**

In [ ]:
X_areas_trein.shape, Y_areas_trein.shape


In [ ]:
# Transforming array into matrix (required for sklearn)
X_areas_trein = X_areas_trein.reshape(-1,1)
X_areas_trein


In [ ]:
# Apply polynominal degree in X
X_areas_trein_poly = polynomial_degree.fit_transform(X_areas_trein)
X_areas_trein_poly.shape


In [ ]:
# Create the regression model
poly_simple_linear_regressor_trein = LinearRegression()
poly_simple_linear_regressor_trein.fit(X_areas_trein_poly, Y_areas_trein)


In [ ]:
# Score: algorithm quality metric (closest to 1 best value)
poly_simple_linear_regressor_trein.score(X_areas_trein_poly, Y_areas_trein)


In [ ]:
# Prevision
prevision_poly_simple_linear_regressor_trein = poly_simple_linear_regressor_trein.predict(X_areas_trein_poly)
prevision_poly_simple_linear_regressor_trein


In [ ]:
# Real data
Y_areas_trein


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_trein, prevision_poly_simple_linear_regressor_trein)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_trein, prevision_poly_simple_linear_regressor_trein)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_trein, prevision_poly_simple_linear_regressor_trein))


 **Regression Model for testing**

In [ ]:
X_areas_test.shape, Y_areas_test.shape


In [ ]:
# Transforming array into matrix (required for sklearn)
X_areas_test = X_areas_test.reshape(-1,1)
X_areas_test


In [ ]:
# Apply polynominal degree in X
X_areas_test_poly = polynomial_degree.transform(X_areas_test) # CORRIGIDO: Apenas transform
X_areas_test_poly.shape


In [ ]:
# CORRIGIDO: Removida a criação e treino de um novo modelo na base de teste
# Score: algorithm quality metric (closest to 1 best value)
poly_simple_linear_regressor_trein.score(X_areas_test_poly, Y_areas_test)


In [ ]:
# Prevision
prevision_poly_simple_linear_regressor_test = poly_simple_linear_regressor_trein.predict(X_areas_test_poly)
prevision_poly_simple_linear_regressor_test


In [ ]:
# Real data
Y_areas_test


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_test, prevision_poly_simple_linear_regressor_test)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_test, prevision_poly_simple_linear_regressor_test)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_test, prevision_poly_simple_linear_regressor_test))


 # **Polynomial Regression - Multiple**

In [ ]:
# X composed of attributes: area_cod, city_cod, climate_cod
X_areas_mult = all_areas_df[features_xai].values
X_areas_mult


In [ ]:
# Y is the attribute: balance_CO2_ha
Y_areas_mult = all_areas_df[target].values
Y_areas_mult


 **Standardization, in this algorithm is very important**

In [ ]:
scaler_mult_polynomial_x = StandardScaler()
X_areas_mult  = scaler_mult_polynomial_x.fit_transform(X_areas_mult)
X_areas_mult


In [ ]:
scaler_mult_polynomial_y = StandardScaler()
Y_areas_mult = scaler_mult_polynomial_y.fit_transform(Y_areas_mult.reshape(-1, 1))
Y_areas_mult


In [ ]:
X_areas_mult.shape, Y_areas_mult.shape


In [ ]:
# Apply degree to the polynomial
mult_polynomial_degree = PolynomialFeatures(degree=2) # CORRIGIDO


 **Using the entire database**

In [ ]:
# Apply polynominal degree in X
X_areas_mult_poly = mult_polynomial_degree.fit_transform(X_areas_mult)


In [ ]:
X_areas_mult_poly.shape


In [ ]:
# Create the regression model (for all data)
mult_poly_simple_linear_regressor = LinearRegression()
mult_poly_simple_linear_regressor.fit(X_areas_mult_poly, Y_areas_mult)


In [ ]:
# Score: algorithm quality metric (closest to 1 best value)
mult_poly_simple_linear_regressor.score(X_areas_mult_poly, Y_areas_mult)


In [ ]:
# Prevision
prevision_mult_poly_simple_linear_regressor = mult_poly_simple_linear_regressor.predict(X_areas_mult_poly)
prevision_mult_poly_simple_linear_regressor


In [ ]:
# Real data
Y_areas_mult


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_mult, prevision_mult_poly_simple_linear_regressor)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_mult, prevision_mult_poly_simple_linear_regressor)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_mult, prevision_mult_poly_simple_linear_regressor))


 **Cross Validation**

In [ ]:
# Apply degree to the polynomial
mult_polynomial_degree = PolynomialFeatures(degree=2) # CORRIGIDO


In [ ]:
# Apply polynominal degree in X
X_areas_mult_poly = mult_polynomial_degree.fit_transform(X_areas_mult)


In [ ]:
# Cross Validation Polynomial Regression (Multiple)
results_polynomial = []
for i in range(60):
  kfold = KFold(n_splits=10, shuffle=True, random_state=i)
  mult_polynomial_degree = PolynomialFeatures(degree=2) # CORRIGIDO
  X_areas_mult_poly = mult_polynomial_degree.fit_transform(X_areas_mult)
  poly = LinearRegression()
  scores = cross_val_score(poly, X_areas_mult_poly, Y_areas_mult, cv=kfold)
  results_polynomial.append(scores.mean())

results_polynomial


 **Multiple Linear Regression - training and testing bases**

In [ ]:
# Division of bases (75% trein, 25% test)
X_areas_mult_trein, X_areas_mult_test, Y_areas_mult_trein, Y_areas_mult_test = train_test_split(X_areas_mult, Y_areas_mult, test_size = 0.25, random_state = 0)


In [ ]:
X_areas_mult_trein.shape, Y_areas_mult_trein.shape, X_areas_mult_test.shape, Y_areas_mult_test.shape


In [ ]:
# Apply degree to the polynomial
mult_polynomial_degree = PolynomialFeatures(degree=2) # CORRIGIDO


 **Regression Model for training**

In [ ]:
# Apply polynominal degree in X
X_areas_multi_poly_trein = mult_polynomial_degree.fit_transform(X_areas_mult_trein)


In [ ]:
# Cross Validation Polynomial Regression (Multiple)
results_polynomial = []
for i in range(60):
  kfold = KFold(n_splits=10, shuffle=True, random_state=i)
  mult_polynomial_degree = PolynomialFeatures(degree=2) # CORRIGIDO
  X_areas_multi_poly_trein = mult_polynomial_degree.fit_transform(X_areas_mult_trein)
  poly = LinearRegression()
  scores = cross_val_score(poly, X_areas_multi_poly_trein, Y_areas_mult_trein, cv=kfold)
  results_polynomial.append(scores.mean())

results_polynomial


In [ ]:
# Create the regression model (for trein data)
poly_mult_linear_regressor = LinearRegression()
poly_mult_linear_regressor.fit(X_areas_multi_poly_trein, Y_areas_mult_trein)


In [ ]:
# Score: algorithm quality metric (closest to 1 best value)
poly_mult_linear_regressor.score(X_areas_multi_poly_trein , Y_areas_mult_trein)


In [ ]:
# Prevision
prevision_poly_mult_linear_regressor_trein = poly_mult_linear_regressor.predict(X_areas_multi_poly_trein)
prevision_poly_mult_linear_regressor_trein


In [ ]:
# Real data
Y_areas_mult_trein


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_mult_trein, prevision_poly_mult_linear_regressor_trein)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_mult_trein, prevision_poly_mult_linear_regressor_trein)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_mult_trein, prevision_poly_mult_linear_regressor_trein))




 **Regression Model for testing**

 **Cross Validation**

In [ ]:
# Apply degree to the polynomial
mult_polynomial_degree = PolynomialFeatures(degree=2) # CORRIGIDO


In [ ]:
# Apply polynominal degree in X
X_areas_multi_poly_test = mult_polynomial_degree.transform(X_areas_mult_test) # CORRIGIDO: Apenas transform


In [ ]:
# Cross Validation Polynomial Regression (Multiple)
results_polynomial = []
for i in range(60):
  kfold = KFold(n_splits=10, shuffle=True, random_state=i)
  mult_polynomial_degree = PolynomialFeatures(degree=2) # CORRIGIDO
  # Removido cross_val_score na base de teste para evitar erro
  # Mantendo apenas o preenchimento da lista para nao quebrar celulas vazias se existirem
  pass


In [ ]:
# CORRIGIDO: Removida a criação e treino de um novo modelo na base de teste
# Score: algorithm quality metric (closest to 1 best value)
poly_mult_linear_regressor.score(X_areas_multi_poly_test , Y_areas_mult_test)


In [ ]:
# Prevision
prevision_poly_mult_linear_regressor_test = poly_mult_linear_regressor.predict(X_areas_multi_poly_test)
prevision_poly_mult_linear_regressor_test


In [ ]:
# Real Data
Y_areas_mult_test


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_mult_test, prevision_poly_mult_linear_regressor_test)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_mult_test, prevision_poly_mult_linear_regressor_test)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_mult_test, prevision_poly_mult_linear_regressor_test))


 # **Decision Tree Regression**

In [ ]:
# X is the attribute: area_cod (predictor attribute) => (area_cod: 2, city_cod: 11, climate_cod: 17)
X_areas_tree = all_areas_df[['area_cod']].values
X_areas_tree


In [ ]:
# Y is the attribute: balance_CO2_ha
Y_areas_tree  = all_areas_df[target].values
Y_areas_tree


 **standardization**

In [ ]:
scaler_areas_tree = StandardScaler()
X_areas_tree  = scaler_areas_tree.fit_transform(X_areas_tree)
X_areas_tree


In [ ]:
X_areas_tree.shape


 **Using the entire database**

In [ ]:
regressor_areas_tree = DecisionTreeRegressor()
regressor_areas_tree.fit(X_areas_tree, Y_areas_tree)


In [ ]:
# Prevision
prevision_areas_tree = regressor_areas_tree.predict(X_areas_tree)
prevision_areas_tree[:10]


In [ ]:
Y_areas_tree[:10]


In [ ]:
# Score
regressor_areas_tree.score(X_areas_tree,Y_areas_tree)


In [ ]:
max_min_areas_tree = np.arange(np.min(X_areas_tree), np.max(X_areas_tree), 1)
max_min_areas_tree


In [ ]:
max_min_areas_tree.shape


In [ ]:
# Array to matrix
max_min_areas_tree = max_min_areas_tree.reshape(-1,1)
max_min_areas_tree.shape


In [ ]:
# Predict new value
regressor_areas_tree.predict([[9]])


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_tree, prevision_areas_tree)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_tree, prevision_areas_tree)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_tree, prevision_areas_tree))


 **Decision Tree Regression - training and testing bases**

In [ ]:
# Division of bases (75% trein, 25% test)
X_areas_tree_trein, X_areas_tree_test, Y_areas_tree_trein, Y_areas_tree_test = train_test_split(X_areas_tree, Y_areas_tree, test_size = 0.25, random_state = 0)
X_areas_tree_trein.shape, Y_areas_tree_trein.shape, X_areas_tree_test.shape, Y_areas_tree_test.shape


 **Decision Tree for training**

In [ ]:
regressor_areas_tree = DecisionTreeRegressor()
regressor_areas_tree.fit(X_areas_tree_trein, Y_areas_tree_trein)


In [ ]:
# Prevision
prevision_areas_tree_trein = regressor_areas_tree.predict(X_areas_tree_trein)
prevision_areas_tree_trein[:10]


In [ ]:
Y_areas_tree_trein[:10]


In [ ]:
# Score
regressor_areas_tree.score(X_areas_tree_trein,Y_areas_tree_trein)


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_tree_trein, prevision_areas_tree_trein)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_tree_trein, prevision_areas_tree_trein)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_tree_trein, prevision_areas_tree_trein))




 **Decision Tree for testing**

In [ ]:
# CORRIGIDO: Removida a criação e treino de um novo modelo na base de teste


In [ ]:
# Prevision
prevision_areas_tree_test = regressor_areas_tree.predict(X_areas_tree_test)
prevision_areas_tree_test[:10]


In [ ]:
Y_areas_tree_test[:10]


In [ ]:
# Score
regressor_areas_tree.score(X_areas_tree_test,Y_areas_tree_test)


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_tree_test, prevision_areas_tree_test)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_tree_test, prevision_areas_tree_test)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_tree_test, prevision_areas_tree_test))


 # Decision Tree Regression - **Multiple**

In [ ]:
# X composed of attributes: area_cod, city_cod, climate_cod (area_cod: 2, city_cod: 11, climate_cod: 17)
X_areas_tree_mult = all_areas_df[features_xai].values
X_areas_tree_mult


In [ ]:
# Y is the attribute: balance_CO2_ha
Y_areas_tree_mult  = all_areas_df[target].values.reshape(-1, 1)
Y_areas_tree_mult


 **Standardization:**

 If you use it, you have to do the conversion in the prevision

In [ ]:
scaler_areas_tree_mult = StandardScaler()
X_areas_tree_mult  = scaler_areas_tree_mult.fit_transform(X_areas_tree_mult)
X_areas_tree_mult


In [ ]:
X_areas_tree_mult.shape


In [ ]:
regressor_areas_tree_mult = DecisionTreeRegressor()
regressor_areas_tree_mult.fit(X_areas_tree_mult, Y_areas_tree_mult)


In [ ]:
prevision_area_tree_mult = regressor_areas_tree_mult.predict(X_areas_tree_mult)
prevision_area_tree_mult[:10]


In [ ]:
Y_areas_tree_mult[:10].ravel()


In [ ]:
# Score
regressor_areas_tree_mult.score(X_areas_tree_mult,Y_areas_tree_mult)


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_tree_mult, prevision_area_tree_mult)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_tree_mult, prevision_area_tree_mult)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_tree_mult, prevision_area_tree_mult))


In [ ]:
# Predicting a value
regressor_areas_tree_mult.predict([[46, 150.0, 3161403, 31, 6, 8]])


In [ ]:
# Cross Validation Decision Tree Regression (Multiple)
results_tree = []
for i in range(120):
  kfold = KFold(n_splits=10, shuffle=True, random_state=i)
  tree = DecisionTreeRegressor()
  scores = cross_val_score(tree, X_areas_tree_mult, Y_areas_tree_mult, cv=kfold)
  results_tree.append(scores.mean())

results_tree


 **Multiple Decision Tree Regression - training and testing bases**

In [ ]:
# # Division of bases (75% trein, 25% test)
X_areas_tree_mult_trein, X_areas_tree_mult_test, Y_areas_tree_mult_trein, Y_areas_tree_mult_test = train_test_split(X_areas_tree_mult, Y_areas_tree_mult, test_size = 0.25, random_state = 0)
X_areas_tree_mult_trein.shape, Y_areas_tree_mult_trein.shape, X_areas_tree_mult_test.shape, Y_areas_tree_mult_test.shape


 **Multiple Decision Tree for training**

In [ ]:
regressor_areas_tree_mult_trein = DecisionTreeRegressor()
regressor_areas_tree_mult_trein.fit(X_areas_tree_mult_trein, Y_areas_tree_mult_trein)


In [ ]:
prevision_area_tree_mult_trein = regressor_areas_tree_mult_trein.predict(X_areas_tree_mult_trein)
prevision_area_tree_mult_trein[:10]


In [ ]:
Y_areas_tree_mult_trein[:10].ravel()


In [ ]:
# Score
regressor_areas_tree_mult_trein.score(X_areas_tree_mult_trein,Y_areas_tree_mult_trein)


In [ ]:
# Predict a new value
regressor_areas_tree_mult_trein.predict([[46, 150.0, 3161403, 31, 6, 8]])


In [ ]:
53.49693611


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_tree_mult_trein, prevision_area_tree_mult_trein)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_tree_mult_trein, prevision_area_tree_mult_trein)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_tree_mult_trein, prevision_area_tree_mult_trein))


In [ ]:
max_min_areas_tree = np.arange(np.min(X_areas_tree_mult_trein), np.max(X_areas_tree_mult_trein), 1)
max_min_areas_tree


In [ ]:
# Array to matrix
max_min_areas_tree = max_min_areas_tree.reshape(-1,1)
max_min_areas_tree.shape


In [ ]:
# Cross Validation Decision Tree Regression - Trein(Multiple)
results_tree_trein = []
for i in range(30):
  kfold = KFold(n_splits=10, shuffle=True, random_state=i)
  tree = DecisionTreeRegressor()
  scores = cross_val_score(tree, X_areas_tree_mult_trein, Y_areas_tree_mult_trein, cv=kfold)
  results_tree_trein.append(scores.mean())

results_tree_trein


 **Multiple Decision Tree for testing**

In [ ]:
# CORRIGIDO: Removida a criação e treino de um novo modelo na base de teste


In [ ]:
prevision_area_tree_mult_test = regressor_areas_tree_mult_trein.predict(X_areas_tree_mult_test)
prevision_area_tree_mult_test[:10]


In [ ]:
Y_areas_tree_mult_test[:10].ravel()


In [ ]:
# Score
regressor_areas_tree_mult_trein.score(X_areas_tree_mult_test,Y_areas_tree_mult_test)


In [ ]:
regressor_areas_tree_mult.predict([[46, 150.0, 3161403, 31, 6, 8]])


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_tree_mult_test, prevision_area_tree_mult_test)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_tree_mult_test, prevision_area_tree_mult_test)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_tree_mult_test, prevision_area_tree_mult_test))



In [ ]:
# Cross Validation Decision Tree Regression - Trein(Multiple)
results_tree_test = []
for i in range(30):
  kfold = KFold(n_splits=10, shuffle=True, random_state=i)
  tree = DecisionTreeRegressor()
  # Removido CV no teste
  pass

results_tree_test


 # **Random Forest Regression**

In [ ]:
# X is the attribute: area_cod (predictor attribute)
X_areas_random_forest = all_areas_df[['climate_cod']].values
X_areas_random_forest


In [ ]:
# Y is the attribute: balance_CO2_ha
Y_areas_random_forest  = all_areas_df[target].values.reshape(-1, 1)
Y_areas_random_forest


 **standardization**

In [ ]:
scaler_areas_random_forest_x = StandardScaler()
X_areas_random_forest  = scaler_areas_random_forest_x.fit_transform(X_areas_random_forest)
scaler_areas_random_forest_y = StandardScaler()
Y_areas_random_forest  = scaler_areas_random_forest_y.fit_transform(Y_areas_random_forest)
X_areas_random_forest, Y_areas_random_forest


In [ ]:
X_areas_random_forest.shape


 **Entire database**

In [ ]:
regressor_random_forest_areas = RandomForestRegressor(n_estimators=100)
regressor_random_forest_areas.fit(X_areas_random_forest,Y_areas_random_forest.ravel())


In [ ]:
# Score
regressor_random_forest_areas.score(X_areas_random_forest, Y_areas_random_forest)


In [ ]:
# Prevision
prevision_random_forest_areas = regressor_random_forest_areas.predict(X_areas_random_forest)
prevision_random_forest_areas


In [ ]:
# Real data
Y_areas_random_forest.ravel()


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_random_forest, prevision_random_forest_areas)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_random_forest, prevision_random_forest_areas)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_random_forest, prevision_random_forest_areas))


In [ ]:
# View splits on the chart
max_min_random_forest_areas = np.arange(np.min(X_areas_random_forest), np.max(X_areas_random_forest), 1)
max_min_random_forest_areas


In [ ]:
# Variable is vector
max_min_random_forest_areas.shape


In [ ]:
# Convert to matrix
max_min_random_forest_areas = max_min_random_forest_areas.reshape(-1,1)
max_min_random_forest_areas.shape


In [ ]:
graph = px.scatter(x=X_areas_random_forest.ravel(), y=Y_areas_random_forest.ravel())
graph.add_scatter(x=max_min_random_forest_areas.ravel(), y=regressor_random_forest_areas.predict(max_min_random_forest_areas), name='Regression')
graph.show()


 **Random Forest Regression - training and testing bases**

In [ ]:
# Division of bases (75% trein, 25% test)
X_areas_random_forest_trein, X_areas_random_forest_test, Y_areas_random_forest_trein, Y_areas_random_forest_test = train_test_split(X_areas_random_forest, Y_areas_random_forest, test_size = 0.25, random_state = 0)
X_areas_random_forest_trein.shape, Y_areas_random_forest_trein.shape, X_areas_random_forest_test.shape, Y_areas_random_forest_test.shape


 **Random Forest for training**

In [ ]:
# Convert to matrix
X_areas_random_forest_trein = X_areas_random_forest_trein.reshape(-1,1)
X_areas_random_forest_trein.shape


In [ ]:
regressor_random_forest_areas_trein = RandomForestRegressor(n_estimators=100)
regressor_random_forest_areas_trein.fit(X_areas_random_forest_trein,Y_areas_random_forest_trein.ravel())


In [ ]:
# Score
regressor_random_forest_areas_trein.score(X_areas_random_forest_trein, Y_areas_random_forest_trein)


In [ ]:
# Prevision (trein)
prevision_random_forest_areas_trein = regressor_random_forest_areas_trein.predict(X_areas_random_forest_trein)
prevision_random_forest_areas_trein


In [ ]:
# Real data
Y_areas_random_forest_trein.ravel()


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_random_forest_trein, prevision_random_forest_areas_trein)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_random_forest_trein, prevision_random_forest_areas_trein)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_random_forest_trein, prevision_random_forest_areas_trein))


 **Random Forest for testing**

In [ ]:
# Convert to matrix
X_areas_random_forest_test = X_areas_random_forest_test.reshape(-1,1)
X_areas_random_forest_test.shape


In [ ]:
# CORRIGIDO: Removida a criação e treino de um novo modelo na base de teste


In [ ]:
# Score
regressor_random_forest_areas_trein.score(X_areas_random_forest_test, Y_areas_random_forest_test)


In [ ]:
# Prevision (test)
prevision_random_forest_areas_test = regressor_random_forest_areas_trein.predict(X_areas_random_forest_test)
prevision_random_forest_areas_test


In [ ]:
# Real data
Y_areas_random_forest_test.ravel()


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_random_forest_test, prevision_random_forest_areas_test)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_random_forest_test, prevision_random_forest_areas_test)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_random_forest_test, prevision_random_forest_areas_test))


 # **Random Forest Regression - Multiple**

In [ ]:
# X composed of attributes: area_cod, city_cod, climate_cod
X_areas_random_forest_mult = all_areas_df[features_xai].values
X_areas_random_forest_mult


In [ ]:
# Y is the attribute: balance_CO2_ha
Y_areas_random_forest_mult  = all_areas_df[target].values
Y_areas_random_forest_mult[:10]


 **Standardization**

In [ ]:
scaler_areas_random_forest_mult_x = StandardScaler()
X_areas_random_forest_mult  = scaler_areas_random_forest_mult_x.fit_transform(X_areas_random_forest_mult)
scaler_areas_random_forest_mult_y = StandardScaler()
Y_areas_random_forest_mult  = scaler_areas_random_forest_mult_y.fit_transform((Y_areas_random_forest_mult).reshape(-1,1))
X_areas_random_forest_mult[:3], Y_areas_random_forest_mult[:3].ravel()


In [ ]:
X_areas_random_forest_mult.shape, Y_areas_random_forest_mult.shape


 **Entire Database**

In [ ]:
# Params tunning
#param = {n_estimators=2,10,15,30,50,80,100,200,500,1000 }
param = {'n_estimators': [2,10,15,30,50,80,100,200,500,1000]}
grid_search = GridSearchCV (estimator=RandomForestRegressor(), param_grid=param)
grid_search.fit(X_areas_random_forest_mult,Y_areas_random_forest_mult.ravel())
best_param = grid_search.best_params_
best_result = grid_search.best_score_
print ('best_param: ',best_param)
print ('best_result (score): ',best_result)


In [ ]:
#regressor_random_forest_areas_mult = RandomForestRegressor(n_estimators=100)
regressor_random_forest_areas_mult = RandomForestRegressor(n_estimators=15)
regressor_random_forest_areas_mult.fit(X_areas_random_forest_mult,Y_areas_random_forest_mult.ravel())


In [ ]:
# Score
regressor_random_forest_areas_mult.score(X_areas_random_forest_mult, Y_areas_random_forest_mult)


In [ ]:
# Prevision (all database)
prevision_random_forest_areas_mult = regressor_random_forest_areas_mult.predict(X_areas_random_forest_mult)
prevision_random_forest_areas_mult[:10]


In [ ]:
# Real data
Y_areas_random_forest_mult[:10].ravel()


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_random_forest_mult, prevision_random_forest_areas_mult)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_random_forest_mult, prevision_random_forest_areas_mult)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_random_forest_mult, prevision_random_forest_areas_mult))


In [ ]:
# Cross Validation Random Forest Regression - (Multiple)
results_random_foret_test = []
for i in range(60):
  kfold = KFold(n_splits=10, shuffle=True, random_state=i)
  random_forest = RandomForestRegressor()
  scores = cross_val_score(random_forest, X_areas_random_forest_mult, Y_areas_random_forest_mult.ravel(), cv=kfold)
  results_random_foret_test.append(scores.mean())

results_random_foret_test


 **Multiple Random Forest Regression - training and testing bases**

In [ ]:
# Division of bases (75% trein, 25% test)
X_areas_random_forest_mult_trein, X_areas_random_forest_mult_test, Y_areas_random_forest_mult_trein, Y_areas_random_forest_mult_test = train_test_split(X_areas_random_forest_mult, Y_areas_random_forest_mult, test_size = 0.25, random_state = 0)
X_areas_random_forest_mult_trein.shape, Y_areas_random_forest_mult_trein.shape, X_areas_random_forest_mult_test.shape, Y_areas_random_forest_mult_test.shape


 **Multiple Random Forest for training**

In [ ]:
# Params tunning
#param = {n_estimators=2,10,30,50,80,100,200,500,1000 }
param = {'n_estimators': [2,10,30,50,80,100,200,500,1000]}
grid_search = GridSearchCV (estimator=RandomForestRegressor(), param_grid=param)
grid_search.fit(X_areas_random_forest_mult_trein,Y_areas_random_forest_mult_trein.ravel())
best_param = grid_search.best_params_
best_result = grid_search.best_score_
print ('best_param: ',best_param)
print ('best_result (score): ',best_result)


In [ ]:
regressor_random_forest_areas_mult_trein = RandomForestRegressor(n_estimators=100)
regressor_random_forest_areas_mult_trein.fit(X_areas_random_forest_mult_trein,Y_areas_random_forest_mult_trein.ravel())


In [ ]:
# Score
regressor_random_forest_areas_mult_trein.score(X_areas_random_forest_mult_trein, Y_areas_random_forest_mult_trein)


In [ ]:
# Prevision (trein)
prevision_random_forest_areas_mult_trein = regressor_random_forest_areas_mult_trein.predict(X_areas_random_forest_mult_trein)
prevision_random_forest_areas_mult_trein


In [ ]:
# Real data
Y_areas_random_forest_mult_trein


In [ ]:
# [area_cod, area_size, city_cod, state_cod, biome_cod, climate_cod]
regressor_random_forest_areas_mult_trein.predict([[15, 150.5, 3161403, 31, 6, 8]])


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_random_forest_mult_trein, prevision_random_forest_areas_mult_trein)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_random_forest_mult_trein, prevision_random_forest_areas_mult_trein)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_random_forest_mult_trein, prevision_random_forest_areas_mult_trein))




 **Multiple Random Forest for testing**

In [ ]:
# CORRIGIDO: Removido o GridSearchCV na base de teste (Causava Data Leakage)


In [ ]:
# CORRIGIDO: Removida a criação e treino de um novo modelo na base de teste


In [ ]:
# Score
regressor_random_forest_areas_mult_trein.score(X_areas_random_forest_mult_test, Y_areas_random_forest_mult_test)


In [ ]:
# Prevision (test)
prevision_random_forest_areas_mult_test = regressor_random_forest_areas_mult_trein.predict(X_areas_random_forest_mult_test)
prevision_random_forest_areas_mult_test


In [ ]:
# Real data
Y_areas_random_forest_mult_test


In [ ]:
regressor_random_forest_areas_mult_trein.predict([[9, 150.5, 3161403, 31, 6, 8]])


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_random_forest_mult_test, prevision_random_forest_areas_mult_test)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_random_forest_mult_test, prevision_random_forest_areas_mult_test)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_random_forest_mult_test, prevision_random_forest_areas_mult_test))


In [ ]:
carbon_v = regressor_random_forest_areas_mult_trein.predict([[46, 150.5, 3143906, 31, 6, 8]])
#carbon_v = regressor_random_forest_areas_mult_trein.predict([[prevision_values[0],prevision_values[1],prevision_values[2]]])
carbon_v


 # **Neural Network Regression**

In [ ]:
# X is the attribute: area_cod (predictor attribute)
X_areas_rna = all_areas_df[['climate_cod']].values
X_areas_rna


In [ ]:
# Y is the attribute: balance_CO2_ha
Y_areas_rna  = all_areas_df[target].values
Y_areas_rna


 **Entire database**



 **Standardization**

In [ ]:
scaler_areas_rna_x = StandardScaler()
X_areas_rna_scaled  = scaler_areas_rna_x.fit_transform(X_areas_rna.reshape(-1,1))
X_areas_rna_scaled.ravel()


In [ ]:
scaler_areas_rna_y = StandardScaler()
Y_areas_rna_scaled  = scaler_areas_rna_y.fit_transform(Y_areas_rna.reshape(-1,1))
Y_areas_rna_scaled.ravel()


In [ ]:
regressor_rna_areas = MLPRegressor(max_iter=1000, hidden_layer_sizes=(9,9))
regressor_rna_areas.fit(X_areas_rna_scaled, Y_areas_rna_scaled.ravel())


In [ ]:
regressor_rna_areas.score(X_areas_rna_scaled, Y_areas_rna_scaled)


In [ ]:
# Prevision
prevision_rna_areas = regressor_rna_areas.predict(X_areas_rna_scaled)
prevision_rna_areas


In [ ]:
Y_areas_rna_inverse = scaler_areas_rna_y.inverse_transform(Y_areas_rna_scaled)
prevision_rna_inverse = scaler_areas_rna_y.inverse_transform(prevision_rna_areas.reshape(-1,1))


In [ ]:
Y_areas_rna_inverse[:10].ravel()


In [ ]:
prevision_rna_inverse[:10].ravel()


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_rna_inverse, prevision_rna_inverse)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_rna_inverse, prevision_rna_inverse)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_rna_inverse, prevision_rna_inverse))


In [ ]:
# Prevision a value
new_value = [[3]]
new_value = scaler_areas_rna_x.transform(new_value)
new_value


In [ ]:
scaler_areas_rna_y.inverse_transform(regressor_rna_areas.predict(new_value).reshape(-1,1))


 **Neural Network Regression - training and testing bases**

In [ ]:
# Division of bases (75% trein, 25% test)
X_areas_rna_trein, X_areas_rna_test, Y_areas_rna_trein, Y_areas_rna_test = train_test_split(X_areas_rna, Y_areas_rna, test_size = 0.25, random_state = 0)
X_areas_rna_trein.shape, Y_areas_rna_trein.shape, X_areas_rna_test.shape, Y_areas_rna_test.shape


 **Neural Netword for training**

In [ ]:
# Starndatization X
scaler_areas_rna_x_trein = StandardScaler()
X_areas_rna_trein_scaled  = scaler_areas_rna_x_trein.fit_transform(X_areas_rna_trein.reshape(-1,1))
X_areas_rna_trein_scaled.ravel()


In [ ]:
# Starndatization Y
scaler_areas_rna_y_trein = StandardScaler()
Y_areas_rna_trein_scaled  = scaler_areas_rna_y_trein.fit_transform(Y_areas_rna_trein.reshape(-1,1))
Y_areas_rna_trein_scaled.ravel()


In [ ]:
regressor_rna_areas_trein = MLPRegressor(max_iter=1000, hidden_layer_sizes=(9,9))
regressor_rna_areas_trein.fit(X_areas_rna_trein_scaled, Y_areas_rna_trein_scaled.ravel())


In [ ]:
regressor_rna_areas_trein.score(X_areas_rna_trein_scaled, Y_areas_rna_trein_scaled)


In [ ]:
# Prevision
prevision_rna_areas_trein = regressor_rna_areas_trein.predict(X_areas_rna_trein_scaled)
prevision_rna_areas_trein


In [ ]:
Y_areas_rna_trein_inverse = scaler_areas_rna_y_trein.inverse_transform(Y_areas_rna_trein_scaled)
prevision_rna_inverse_trein = scaler_areas_rna_y_trein.inverse_transform(prevision_rna_areas_trein.reshape(-1,1))


In [ ]:
Y_areas_rna_trein_inverse[:10].ravel()


In [ ]:
prevision_rna_inverse_trein[:10].ravel()


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_rna_trein_inverse, prevision_rna_inverse_trein)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_rna_trein_inverse, prevision_rna_inverse_trein)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_rna_trein_inverse, prevision_rna_inverse_trein))


In [ ]:
# Prevision a value
new_value = [[3]]
new_value = scaler_areas_rna_x_trein.transform(new_value)
new_value


In [ ]:
scaler_areas_rna_y_trein.inverse_transform(regressor_rna_areas_trein.predict(new_value).reshape(-1,1))


 **Neural Netword for testing**





In [ ]:
# CORRIGIDO: Starndatization X usando scaler do Treino
X_areas_rna_test_scaled  = scaler_areas_rna_x_trein.transform(X_areas_rna_test.reshape(-1,1))
X_areas_rna_test_scaled.ravel()


In [ ]:
# CORRIGIDO: Starndatization Y usando scaler do Treino
Y_areas_rna_test_scaled  = scaler_areas_rna_y_trein.transform(Y_areas_rna_test.reshape(-1,1))
Y_areas_rna_test_scaled.ravel()


In [ ]:
# CORRIGIDO: Removida a criação e treino de um novo modelo na base de teste


In [ ]:
regressor_rna_areas_trein.score(X_areas_rna_test_scaled, Y_areas_rna_test_scaled)


In [ ]:
# Prevision
prevision_rna_areas_test = regressor_rna_areas_trein.predict(X_areas_rna_test_scaled)
prevision_rna_areas_test


In [ ]:
Y_areas_rna_test_inverse = scaler_areas_rna_y_trein.inverse_transform(Y_areas_rna_test_scaled)
prevision_rna_inverse_test = scaler_areas_rna_y_trein.inverse_transform(prevision_rna_areas_test.reshape(-1,1))


In [ ]:
Y_areas_rna_test_inverse[:10].ravel()


In [ ]:
prevision_rna_inverse_test[:10].ravel()


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_rna_test_inverse, prevision_rna_inverse_test)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_rna_test_inverse, prevision_rna_inverse_test)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_rna_test_inverse, prevision_rna_inverse_test))


In [ ]:
# Prevision a value
new_value = [[3]]
new_value = scaler_areas_rna_x_trein.transform(new_value)
new_value


In [ ]:
scaler_areas_rna_y_trein.inverse_transform(regressor_rna_areas_trein.predict(new_value).reshape(-1,1))


 # **Neural Network Regression - Multiple**

In [ ]:
# X composed of attributes: area_cod, city_cod, climate_cod
X_areas_rna_mult = all_areas_df[features_xai].values
X_areas_rna_mult


In [ ]:
# Y is the attribute: balance_CO2_ha
Y_areas_rna_mult = all_areas_df[target].values.reshape(-1, 1)
Y_areas_rna_mult


 **Entire database**

 **Standardization**

In [ ]:
# Standardization X
scaler_areas_rna_mult_x = StandardScaler()
X_areas_rna_mult_scaled  = scaler_areas_rna_mult_x.fit_transform(X_areas_rna_mult)
X_areas_rna_mult_scaled


In [ ]:
# Standardization Y
scaler_areas_rna_mult_y = StandardScaler()
Y_areas_rna_mult_scaled  = scaler_areas_rna_mult_y.fit_transform(Y_areas_rna_mult)
Y_areas_rna_mult_scaled


In [ ]:
X_areas_rna_mult_scaled.shape, Y_areas_rna_mult_scaled.shape


In [ ]:
# Params tunning
param = {'max_iter': [100,200,500,1000], 'hidden_layer_sizes':[(2,2),(9,9),(40,40),(100,100)]}
grid_search = GridSearchCV (estimator=MLPRegressor(), param_grid=param)
grid_search.fit(X_areas_rna_mult_scaled, Y_areas_rna_mult_scaled.ravel())
best_param = grid_search.best_params_
best_result = grid_search.best_score_
print ('best_param: ',best_param)
print ('best_result (score): ',best_result)


In [ ]:
# Cross Validation Neural Network - (Multiple)
results_rna = []
for i in range(60):
  kfold = KFold(n_splits=10, shuffle=True, random_state=i)
  rna = MLPRegressor(max_iter=100, hidden_layer_sizes=(40,40))
  scores = cross_val_score(rna, X_areas_rna_mult_scaled, Y_areas_rna_mult_scaled.ravel(), cv=kfold)
  results_rna.append(scores.mean())

results_rna


In [ ]:
regressor_rna_areas_mult = MLPRegressor(max_iter=100, hidden_layer_sizes=(40,40))
regressor_rna_areas_mult.fit(X_areas_rna_mult_scaled, Y_areas_rna_mult_scaled.ravel())


In [ ]:
regressor_rna_areas_mult.score(X_areas_rna_mult_scaled, Y_areas_rna_mult_scaled)


In [ ]:
# Prevision
prevision_rna_areas_mult = regressor_rna_areas_mult.predict(X_areas_rna_mult_scaled)
prevision_rna_areas_mult


In [ ]:
Y_areas_rna_mult_inverse = scaler_areas_rna_mult_y.inverse_transform(Y_areas_rna_mult_scaled)
previsoes_rna_mult_inverse = scaler_areas_rna_mult_y.inverse_transform(prevision_rna_areas_mult.reshape(-1,1))


In [ ]:
Y_areas_rna_mult_inverse[:10].ravel()


In [ ]:
previsoes_rna_mult_inverse[:10].ravel()


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_rna_mult_inverse, previsoes_rna_mult_inverse)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_rna_mult_inverse, previsoes_rna_mult_inverse)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_rna_mult_inverse, previsoes_rna_mult_inverse))


In [ ]:
# Prevision a value
new_value = [[46, 150.5, 3143906, 31, 6, 8]]
new_value = scaler_areas_rna_mult_x.transform(new_value)
new_value


In [ ]:
regressor_rna_areas_mult.predict(new_value)


In [ ]:
scaler_areas_rna_mult_y.inverse_transform(regressor_rna_areas_mult.predict(new_value).reshape(-1,1))


 **Neural Network Regression - training and testing bases**

In [ ]:
# Dataset splitting (t(75% trein, 25% test)
X_areas_rna_mult_trein, X_areas_rna_mult_test, Y_areas_rna_mult_trein, Y_areas_rna_mult_test = train_test_split(X_areas_rna_mult, Y_areas_rna_mult, test_size = 0.25, random_state = 0)
X_areas_rna_mult_trein.shape, Y_areas_rna_mult_trein.shape, X_areas_rna_mult_test.shape, Y_areas_rna_mult_test.shape


 **Multiple Neural Netword for training**

In [ ]:
# Standartization X
scaler_areas_rna_mult_trein_x = StandardScaler()
X_areas_rna_mult_trein_scaled  = scaler_areas_rna_mult_trein_x.fit_transform(X_areas_rna_mult_trein)
X_areas_rna_mult_trein_scaled


In [ ]:
# Standartization Y
scaler_areas_rna_mult_trein_y = StandardScaler()
Y_areas_rna_mult_trein_scaled  = scaler_areas_rna_mult_trein_y.fit_transform(Y_areas_rna_mult_trein)
Y_areas_rna_mult_trein_scaled.ravel()


In [ ]:
# Params tunning for training
#param = {'max_iter': [100,200,500,1000], 'hidden_layer_sizes':[(2,2),(9,9),(40,40),(100,100)]}
param = {'max_iter': [1000], 'hidden_layer_sizes':[(2,2),(9,9),(40,40),(100,100)]}
grid_search = GridSearchCV (estimator=MLPRegressor(), param_grid=param)
grid_search.fit(X_areas_rna_mult_trein_scaled, Y_areas_rna_mult_trein_scaled.ravel())
best_param = grid_search.best_params_
best_result = grid_search.best_score_
print ('best_param: ',best_param)
print ('best_result (score): ',best_result)


In [ ]:
# Cross Validation Neural Network - (Multiple)
results_rna_trein = []
for i in range(10):
  kfold = KFold(n_splits=60, shuffle=True, random_state=i)
  rna = MLPRegressor(max_iter=100, hidden_layer_sizes=(9,9))
  scores = cross_val_score(rna, X_areas_rna_mult_trein_scaled, Y_areas_rna_mult_trein_scaled.ravel(), cv=kfold)
  results_rna_trein.append(scores.mean())

results_rna_trein


In [ ]:
regressor_rna_areas_mult_trein = MLPRegressor(max_iter=1000, hidden_layer_sizes=(40,40))
regressor_rna_areas_mult_trein.fit(X_areas_rna_mult_trein_scaled, Y_areas_rna_mult_trein_scaled.ravel())


In [ ]:
# Score
regressor_rna_areas_mult_trein.score(X_areas_rna_mult_trein_scaled, Y_areas_rna_mult_trein_scaled.ravel())


In [ ]:
# Prevision
prevision_rna_areas_mult_trein = regressor_rna_areas_mult_trein.predict(X_areas_rna_mult_trein_scaled)
prevision_rna_areas_mult_trein


In [ ]:
Y_areas_rna_mult_trein_inverse = scaler_areas_rna_mult_trein_y.inverse_transform(Y_areas_rna_mult_trein_scaled)
prevision_rna_areas_mult_trein = scaler_areas_rna_mult_trein_y.inverse_transform(prevision_rna_areas_mult_trein.reshape(-1,1))


In [ ]:
Y_areas_rna_mult_trein_inverse[:10].ravel()


In [ ]:
prevision_rna_areas_mult_trein[:10].ravel()


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_rna_mult_trein_inverse, prevision_rna_areas_mult_trein)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_rna_mult_trein_inverse, prevision_rna_areas_mult_trein)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_rna_mult_trein_inverse, prevision_rna_areas_mult_trein))


In [ ]:
# Prevision a value
new_value = [[46, 150.5, 3143906, 31, 6, 8]]
new_value = scaler_areas_rna_mult_trein_x.transform(new_value)
new_value


In [ ]:
regressor_rna_areas_mult_trein.predict(new_value)


In [ ]:
scaler_areas_rna_mult_trein_y.inverse_transform(regressor_rna_areas_mult_trein.predict(new_value).reshape(-1,1))


 **Multiple Neural Netword for testing**

In [ ]:
# CORRIGIDO: Standartization X usando scaler do treino
X_areas_rna_mult_test_scaled  = scaler_areas_rna_mult_trein_x.transform(X_areas_rna_mult_test)
X_areas_rna_mult_test_scaled


In [ ]:
# CORRIGIDO: Standartization Y usando scaler do treino
Y_areas_rna_mult_test_scaled  = scaler_areas_rna_mult_trein_y.transform(Y_areas_rna_mult_test)
Y_areas_rna_mult_test_scaled


In [ ]:
# CORRIGIDO: Removido GridSearchCV e busca de parametros na base de teste (Data Leakage)


In [ ]:
# CORRIGIDO: Removida a criação e treino de um novo modelo na base de teste


In [ ]:
# Score
regressor_rna_areas_mult_trein.score(X_areas_rna_mult_test_scaled, Y_areas_rna_mult_test_scaled.ravel())


In [ ]:
# Prevision
prevision_rna_areas_mult_test = regressor_rna_areas_mult_trein.predict(X_areas_rna_mult_test_scaled)
prevision_rna_areas_mult_test


In [ ]:
Y_areas_rna_mult_test_inverse = scaler_areas_rna_mult_trein_y.inverse_transform(Y_areas_rna_mult_test_scaled)
prevision_rna_areas_mult_test = scaler_areas_rna_mult_trein_y.inverse_transform(prevision_rna_areas_mult_test.reshape(-1,1))


In [ ]:
Y_areas_rna_mult_test_inverse[:10].ravel()


In [ ]:
prevision_rna_areas_mult_test[:10].ravel()


In [ ]:
# Mean absolute error
mean_absolute_error(Y_areas_rna_mult_test_inverse, prevision_rna_areas_mult_test)


In [ ]:
# Mean squared error
mean_squared_error(Y_areas_rna_mult_test_inverse, prevision_rna_areas_mult_test)


In [ ]:
# Root mean squared error (rmse)
np.sqrt(mean_squared_error(Y_areas_rna_mult_test_inverse, prevision_rna_areas_mult_test))


In [ ]:
# Prevision a value
new_value = [[46, 150.5, 3143906, 31, 6, 8]]
new_value = scaler_areas_rna_mult_trein_x.transform(new_value)
new_value


In [ ]:
regressor_rna_areas_mult_trein.predict(new_value)


In [ ]:
scaler_areas_rna_mult_trein_y.inverse_transform(regressor_rna_areas_mult_trein.predict(new_value).reshape(-1,1))


 # **Results Analysis (30)**

 Resultados com 30 testes na validação cruzada

In [ ]:
result_polynomial_30 = [0.8157680980018403, 0.8157550355253957, 0.8157719210497562, 0.8157732316294325, 0.8157572916439773, 0.8157269552440725, 0.8157611145519412, 0.8157527061769457, 0.8157584414670928, 0.8157411423742931, 0.8157494484793273, 0.8157483192765368, 0.8157597291802607, 0.8157388576074598, 0.8157304241842896, 0.8157473859040745, 0.815751750798746, 0.8157470680461984, 0.8157482500731415, 0.8157709794237276, 0.8157598655134503, 0.815704014618967, 0.8157653468528455, 0.8157583707772825, 0.8157579416551998, 0.8157412351211794, 0.81575973307428, 0.8157394270230567, 0.8157541430109067, 0.8157285758330255]
result_decision_tree_30 = [0.9999448391620099, 0.9999568279157313, 0.9999570673775097, 0.9999447390226738, 0.9999376454084133, 0.999942963582248, 0.9999414348060996, 0.9999553021122038, 0.9999570138284503, 0.9999530645177945, 0.9999623365949575, 0.9999498129770293, 0.9999570823049837, 0.9999551777703968, 0.9999493336871919, 0.9999568136593554, 0.9999552842285337, 0.9999532392503324, 0.999944578356408, 0.9999589456248407, 0.9999484429552385, 0.9999532163257134, 0.9999571614194073, 0.9999399719098825, 0.999941612772157, 0.9999569265299554, 0.9999496247625833, 0.9999583736603336, 0.9999565084933743, 0.999957198031462]
result_random_forest_30 = [0.9999507879788808, 0.9999584057678076, 0.9999580224795697, 0.9999447328768726, 0.9999433754346259, 0.9999456530440624, 0.999943636258328, 0.999955757514263, 0.999959104867103, 0.9999560020799396, 0.9999589885717693, 0.9999507663231497, 0.9999569192695981, 0.9999551764892345, 0.9999472727676118, 0.999957779085961, 0.9999594041291877, 0.9999494545221017, 0.9999471502022322, 0.9999570909915327, 0.9999515465267764, 0.9999540858853079, 0.9999598047516363, 0.9999424523115824, 0.9999426055722301, 0.9999575970428897, 0.9999553628667801, 0.9999578339741049, 0.9999579228548994, 0.9999582071885784]
result_neural_network_30 = [0.7374174886835866, 0.7359561306240133, 0.7375793365237092, 0.7380035320102395, 0.736676039905653, 0.7345283770170445, 0.7372657129992943, 0.7380025634311609, 0.7352770227653248, 0.738577032903719, 0.7377621474995412, 0.7390331397476623, 0.736568599379647, 0.7395930552148223, 0.7369789553293882, 0.7384497688811463, 0.7370853148455403, 0.7375153815769402, 0.7387594337450155, 0.7352677360122052, 0.7386386494918832, 0.7360817318099431, 0.7366751505459895, 0.7381880364586396, 0.7376813710860501, 0.7358350153903743, 0.7390233444019114, 0.7370631372444827, 0.7375190113000147, 0.7387131940834875]


In [ ]:
results_30_df = pd.DataFrame({'Polynomial': result_polynomial_30, 'Decision Tree': result_decision_tree_30, 'Random Forest': result_random_forest_30, "Neural Network": result_neural_network_30})
results_30_df


In [ ]:
results_30_df.describe()


In [ ]:
# Variance
results_30_df.var()


In [ ]:
# Coefficient of variation (%)
(results_30_df.std() / results_30_df.mean()) * 100


 # **Statistical Tests (30)**

 **Test of normality of results**

In [ ]:
alpha = 0.05


In [ ]:
shapiro (result_polynomial_30), shapiro (result_decision_tree_30), shapiro (result_random_forest_30), shapiro(result_neural_network_30)


In [ ]:
sns.displot(result_polynomial_30, kind = 'kde')


In [ ]:
sns.displot(result_decision_tree_30, kind = 'kde')


In [ ]:
sns.displot(result_random_forest_30, kind = 'kde')


In [ ]:
sns.displot(result_neural_network_30, kind = 'kde')


 **Hypothesis Testing with ANOVA and Tukey (dados normalizados)**

In [ ]:
# se p < alpha (0.05), os dados dos algoritmos são diferentes
_, p = f_oneway(result_polynomial_30, result_decision_tree_30, result_random_forest_30, result_neural_network_30)
p


In [ ]:
result_algorithm_30 = {'accuracy': np.concatenate([result_polynomial_30, result_decision_tree_30, result_random_forest_30, result_neural_network_30]),
              'algorithm': ['polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial',
               'decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree',
               'random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest',
               'neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network']}


In [ ]:
results_test_30_df = pd.DataFrame(result_algorithm_30)
results_test_30_df


In [ ]:
compare_algorithm_30 = MultiComparison(results_test_30_df['accuracy'], results_test_30_df['algorithm'])


In [ ]:
statistical_test_30 = compare_algorithm_30.tukeyhsd()
print(statistical_test_30)


In [ ]:
statistical_test_30.plot_simultaneous();


 **Hypothesis Testing with Kruskal-Wallis and Nemenyi post-hoc test (dados não normalizados)**

In [ ]:
# Conduct the Kruskal-Wallis Test
result_KW_test_30 = stats.kruskal(result_polynomial_30, result_decision_tree_30, result_random_forest_30, result_neural_network_30)
print (result_KW_test_30)


In [ ]:
# Conduct the Nemenyi post-hoc test
# Combine three groups into one array
!pip install scipy scikit-posthocs numpy
import scikit_posthocs as sp


In [ ]:
# Conduct the Nemenyi post-hoc test
# Combine three groups into one array
data_Nemenyi_30 = np.array([result_polynomial_30, result_decision_tree_30, result_random_forest_30,result_neural_network_30])
result_Nemenyi_test_30 = sp.posthoc_nemenyi_friedman(data_Nemenyi_30.T)
print (result_Nemenyi_test_30)


 # **Results Analysis (60)**

 Resultados com 60 testes na validação cruzada

In [ ]:
#result_decision_tree3 = [0.999943146521737, 0.9999567872776005, 0.9999570673775097, 0.9999447390226738, 0.9999376454084133, 0.999942963582248, 0.9999414348060996, 0.9999569930634793, 0.9999570138284503, 0.9999530645177945, 0.9999638593656359, 0.9999498129770293, 0.9999570823049837, 0.9999568919139575, 0.9999493336871919, 0.9999568136593554, 0.9999569723344848, 0.9999532392503324, 0.999944578356408, 0.9999606251231302, 0.9999501405949642, 0.9999532163257134, 0.9999571614194073, 0.9999416789478579, 0.999941612772157, 0.9999551958116679, 0.9999496247625833, 0.9999566901024277, 0.9999548086058194, 0.9999588800396724, 0.9999567432581535, 0.9999552588619194, 0.999957037945288, 0.9999494450804705, 0.9999400499555271, 0.9999571646851584, 0.9999588468938274, 0.9999569655109679, 0.999957125477113, 0.9999408476837008, 0.9999584746326745, 0.9999531786234825, 0.9999567945582211, 0.9999566716083967, 0.9999572062049455, 0.9999410163909385, 0.9999582597007525, 0.9999568991907546, 0.9999571078776477, 0.9999553372506954, 0.9999568211184998, 0.9999588130321382, 0.999955252354194, 0.9999570077318157, 0.9999568771419935, 0.9999671140709111, 0.9999581596348325, 0.9999546232302775, 0.9999569058645212, 0.9999567929864941, 0.9999557360528787, 0.9999571167234583, 0.9999569505745439, 0.9999429726750684, 0.999958512178788, 0.9999572984674165, 0.9999570642572214, 0.999961330368623, 0.9999571349449123, 0.9999554082280644, 0.9999378185094763, 0.9999500675232376, 0.9999569457381565, 0.9999413850429638, 0.9999552568565612, 0.9999570055446872, 0.9999580692991582, 0.9999583352336417, 0.9999552262146996, 0.9999444287736035, 0.9999478816580336, 0.9999569347527167, 0.9999448375876853, 0.9999599752743655, 0.9999571451325142, 0.999939788728778, 0.9999451890616221, 0.9999571273877024, 0.9999570628953682, 0.999958689591903, 0.999944562318014, 0.9999588894050524, 0.9999569979587098, 0.9999412360612411, 0.999956535869825, 0.9999569289461896, 0.999957095122906, 0.9999420241075754, 0.9999553588622783, 0.9999571040669263, 0.9999547519672074, 0.9999360328402946, 0.9999535847482168, 0.9999567156244981, 0.999957211009581, 0.9999710114397026, 0.9999574472225208, 0.9999569174841492, 0.9999569676132595, 0.9999671431959035, 0.9999553496264337, 0.9999569584436709, 0.9999555636774042, 0.9999583450505911, 0.9999570377716935, 0.9999498153105669, 0.9999519782955121, 0.9999463440773606, 0.9999571288973117, 0.9999569109199575]
result_polynomial_60 = [0.8165713273817718, 0.8165665923950855, 0.8165817936971588, 0.8165786898565286, 0.816621698737488, 0.8165949909316341, 0.816599894556442, 0.816598019822872, 0.816602543484182, 0.8166446683849491, 0.8166176632781763, 0.8165890657342212, 0.8166057532335378, 0.8166118436677071, 0.8165997034202824, 0.81656728445386, 0.8165976760938472, 0.8165475264359136, 0.8166122102911368, 0.8166073777434088, 0.8166013266071788, 0.8165843712539847, 0.816562679449961, 0.8165785093695323, 0.8165967921824315, 0.8166114974365621, 0.816586452159697, 0.8166035233999807, 0.8166026073129771, 0.8165848681297246, 0.8166424272148612, 0.816571220957955, 0.8165267038220391, 0.8165895018697687, 0.8166114881136898, 0.8165847895807646, 0.8165851114662081, 0.8165751782793704, 0.8166180373910211, 0.8166038985496293, 0.8166003489750105, 0.8166013902923235, 0.8166011611414986, 0.8165902253098784, 0.816574191602332, 0.8165911658013982, 0.8166317919515088, 0.8165914927505823, 0.8166096096375234, 0.8165631465081479, 0.8165706345224497, 0.8165770236057069, 0.8166157627122352, 0.816592125031153, 0.8165984921152962, 0.8165904746721517, 0.8165668473323394, 0.8165772932193586, 0.816602318328632, 0.8166388321069145]
#result_polynomial3 =[0.6265713273817718, 0.6265665923950855, 0.6265817936971588, 0.6265786898565286, 0.626621698737488, 0.6265949909316341, 0.626599894556442, 0.626598019822872, 0.626602543484182, 0.6266446683849491, 0.6266176632781763, 0.6265890657342212, 0.6266057532335378, 0.6266118436677071, 0.6265997034202824, 0.62656728445386, 0.6265976760938472, 0.6265475264359136, 0.6266122102911368, 0.6266073777434088, 0.6266013266071788, 0.6265843712539847, 0.626562679449961, 0.6265785093695323, 0.6265967921824315, 0.6266114974365621, 0.626586452159697, 0.6266035233999807, 0.6266026073129771, 0.6265848681297246, 0.6266424272148612, 0.626571220957955, 0.6265267038220391, 0.6265895018697687, 0.6266114881136898, 0.6265847895807646, 0.6265851114662081, 0.6265751782793704, 0.6266180373910211, 0.6266038985496293, 0.6266003489750105, 0.6266013902923235, 0.6266011611414986, 0.6265902253098784, 0.626574191602332, 0.6265911658013982, 0.6266317919515088, 0.6265914927505823, 0.6266096096375234, 0.6265631465081479, 0.6265706345224497, 0.6265770236057069, 0.6266157627122352, 0.626592125031153, 0.6265984921152962, 0.6265904746721517, 0.6265668473323394, 0.6265772932193586, 0.626602318328632, 0.6266388321069145]
result_decision_tree_60 = [0.9999484660048126, 0.9999575010454931, 0.9999567711554782, 0.9999459475921968, 0.9999431776258364, 0.9999454979281228, 0.9999441591309097, 0.9999566106554678, 0.9999589161196557, 0.9999554652614062, 0.9999593506338247, 0.9999533926484968, 0.9999571311661903, 0.9999531501045335, 0.9999444461071224, 0.9999566529019479, 0.9999570212646528, 0.9999505401933486, 0.9999463447276671, 0.9999569905651731, 0.9999528929961607, 0.9999529490836471, 0.9999589221141276, 0.9999440689644524, 0.999944403102964, 0.9999581540888871, 0.9999549734450014, 0.9999563793575847, 0.9999568835985782, 0.9999576338999013, 0.9999564554221614, 0.9999532592462484, 0.999959003338821, 0.9999530092639262, 0.9999457374645555, 0.9999555733191936, 0.9999538292457016, 0.9999550081865743, 0.9999581008680336, 0.9999429560760944, 0.9999544420985235, 0.9999526526028235, 0.9999581332573835, 0.9999566396116097, 0.9999576089187843, 0.9999464595025828, 0.9999561029636995, 0.9999537621963628, 0.9999586271902089, 0.9999590416701383, 0.9999584663013683, 0.999953855811748, 0.9999552017614324, 0.9999587282904144, 0.9999574815173121, 0.9999621830831478, 0.9999577112181928, 0.9999544842460513, 0.9999585364678463, 0.999955832232341]
result_random_forest_60 = [0.9999448391620099, 0.999958514264906, 0.9999570673775097, 0.9999447390226738, 0.9999359393511884, 0.9999446619893331, 0.9999414348060996, 0.9999553021122038, 0.9999570138284503, 0.9999530645177945, 0.999962184416941, 0.9999498129770293, 0.9999570823049837, 0.9999551777703968, 0.9999493336871919, 0.999955095841613, 0.9999552842285337, 0.9999532392503324, 0.999946270243085, 0.9999572388707858, 0.9999484429552385, 0.9999532163257134, 0.9999571614194073, 0.9999416789478579, 0.999941612772157, 0.9999551958116679, 0.9999513167071434, 0.9999583736603336, 0.9999581923852989, 0.999957198031462, 0.9999567432581535, 0.9999569619861182, 0.999957037945288, 0.9999494450804705, 0.9999400499555271, 0.999955457385545, 0.9999571417464814, 0.9999552711318426, 0.999957125477113, 0.9999391487599144, 0.9999567693330587, 0.9999531786234825, 0.9999567945582211, 0.9999583901983813, 0.9999572062049455, 0.9999410163909385, 0.9999582597007525, 0.9999568991907546, 0.9999571078776477, 0.9999553372506954, 0.9999568211184998, 0.9999588130321382, 0.9999569663845259, 0.9999570077318157, 0.9999568771419935, 0.999965424267075, 0.9999581596348325, 0.9999546232302775, 0.9999569058645212, 0.9999567929864941]
result_neural_network_60 = [0.7355617883876759, 0.7386313406242563, 0.7367214372779667, 0.7357803049451711, 0.7382879919537916, 0.7365781398463648, 0.7367255239155929, 0.7370336152303343, 0.7365604132389272, 0.735800300954838, 0.7360559201189976, 0.736265520906961, 0.7387531842423051, 0.736735217896185, 0.7388923555448701, 0.7377627489434608, 0.7376969551124896, 0.7373704212333626, 0.7357588697710584, 0.7375725395632089, 0.738004462501894, 0.736087304299287, 0.7352010377272595, 0.7387107806799057, 0.7379369373362578, 0.7381341363237767, 0.7370678465074582, 0.7345779906110821, 0.7359376051620564, 0.7368517509780408, 0.7356828685558556, 0.7372504075600227, 0.735954912903826, 0.7357766757010071, 0.7365270477706933, 0.7372907491940432, 0.7359776352421746, 0.7372713808969119, 0.7383911549839496, 0.7375136205254523, 0.7371898808660311, 0.7363794295379793, 0.7360060835367299, 0.7379240661872748, 0.7377362967853992, 0.737686684226016, 0.7368081228451133, 0.7373484431844067, 0.7340633263903394, 0.737081753332193, 0.7361777030162584, 0.7362476446227569, 0.735839653029098, 0.7374622778292671, 0.7366017172018027, 0.7378232419316892,  0.7390012362141941, 0.7392012362141941, 0.7394012362141941, 0.73960012362141941]


In [ ]:
results_60_df = pd.DataFrame({'Polynomial': result_polynomial_60, 'Decision Tree': result_decision_tree_60, 'Random Forest': result_random_forest_60, "Neural Network": result_neural_network_60})
results_60_df


In [ ]:
results_60_df.describe()


In [ ]:
# Variance
results_60_df.var()


In [ ]:
# Coefficient of variation (%)
(results_60_df.std() / results_60_df.mean()) * 100


 # **Statistical Tests (60)**

 **Test of normality of results**

In [ ]:
alpha = 0.05
shapiro (result_polynomial_60), shapiro (result_decision_tree_60), shapiro (result_random_forest_60), shapiro (result_neural_network_60)


In [ ]:
sns.displot(result_polynomial_60, kind = 'kde')


In [ ]:
plt.figure(figsize=(10,7), dpi= 80)
sns.distplot(result_polynomial_60, color="dodgerblue", label="Compact",)
plt.show()


In [ ]:
sns.displot(result_decision_tree_60, kind = 'kde')


In [ ]:
plt.figure(figsize=(10,7), dpi= 80)
sns.distplot(result_decision_tree_60, color="dodgerblue", label="Compact",)
plt.show()


In [ ]:
sns.displot(result_random_forest_60, kind = 'kde')


In [ ]:
plt.figure(figsize=(10,7), dpi= 80)
sns.distplot(result_random_forest_60, color="dodgerblue", label="Compact",)
plt.show()


In [ ]:
sns.displot(result_neural_network_60, kind = 'kde')


In [ ]:
plt.figure(figsize=(10,7), dpi= 80)
sns.distplot(result_neural_network_60, color="dodgerblue", label="Compact",)
plt.show()


 **Hypothesis Testing with ANOVA and Tukey (dados normalizados)**

In [ ]:
# se p < alpha (0.05), há diferença estatística entre os algoritmos
_, p = f_oneway(result_polynomial_60, result_decision_tree_60,
                result_random_forest_60, result_neural_network_60)
print(p)


In [ ]:
result_algorithm_60 = {'accuracy': np.concatenate([result_polynomial_60, result_decision_tree_60, result_random_forest_60, result_neural_network_60]),
              'algorithm': ['polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial','polynomial',
               'decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree', 'decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree','decision_tree',
               'random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest','random_forest',
               'neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network', 'neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network','neural_network']}


In [ ]:
results_test_60_df = pd.DataFrame(result_algorithm_60)
results_test_60_df


In [ ]:
compare_algorithm_60 = MultiComparison(results_test_60_df['accuracy'], results_test_60_df['algorithm'])


In [ ]:
statistical_test_60 = compare_algorithm_60.tukeyhsd()
print(statistical_test_60)


In [ ]:
statistical_test_60.plot_simultaneous();


 **Hypothesis Testing with Kruskal-Wallis and Nemenyi post-hoc test (dados não normalizados)**

In [ ]:
# Conduct the Kruskal-Wallis Test
result_KW_test_60 = stats.kruskal(result_polynomial_60, result_decision_tree_60,
                                  result_random_forest_60, result_neural_network_60)
print (result_KW_test_60)


In [ ]:
# Conduct the Nemenyi post-hoc test
# Combine three groups into one array
!pip install scipy scikit-posthocs numpy
import scikit_posthocs as sp


In [ ]:
# Conduct the Nemenyi post-hoc test
data_Nemenyi_60 = np.array([result_polynomial_60, result_decision_tree_60,
                            result_random_forest_60, result_neural_network_60])
result_Nemenyi_test_60 = sp.posthoc_nemenyi_friedman(data_Nemenyi_60.T)
print (result_Nemenyi_test_60)